# 02. Data Preprocessing

This notebook includes the data preprocessing steps for both main modelling stages of the project: modelling expected counts and deviations analysis.

## PART 1: Preparing Count Data for Modelling Expected Cyclist Counts


In this part, we prepare the AWV bicycle count data for the first stage of the project: modelling normal cycling traffic in Flanders.

The raw data are provided as monthly CSV files with 15-minute traffic counts. We focus only on cyclist observations (`FIETSERS`) and aggregate the data to 2-hour intervals. This reduces short-term noise while still preserving daily traffic patterns such as morning and evening peaks.

We also keep track of missing count values during aggregation. Missing counts are not treated as zero, because a missing value means that the count is unknown, while zero means that the sensor recorded no cyclists. For each 2-hour interval, we record how many 15-minute observations were available.

After cleaning and aggregation, we add site information and calendar variables such as public holidays and school holidays, as well as fuel prices data.

Important: We include fuel prices in the normal-counts modelling because they are unlikely to explain sudden short-term deviations in one specific 2-hour interval. However, they may influence the general expected level of cycling traffic over weeks or months, so they are treated as part of the expected background conditions.

### Importing packages and defining path


In [1]:
import pandas as pd
from pathlib import Path

project_folder = Path("..")

raw_folder = project_folder / "data" / "raw"
processed_folder = project_folder / "data" / "processed"
external_folder = project_folder / "data" / "external"
diagnostics_folder = processed_folder / "diagnostics"


raw_folder.mkdir(parents=True, exist_ok=True)
processed_folder.mkdir(parents=True, exist_ok=True)
external_folder.mkdir(parents=True, exist_ok=True)
diagnostics_folder.mkdir(parents=True, exist_ok=True)

### File names and colum names

In [2]:
metadata_files = [
    "sites.csv",
    "richtingen.csv",
]

count_files = [
    "data-2022-05.csv",
    "data-2022-06.csv",
    "data-2022-07.csv",
    "data-2022-08.csv",
    "data-2022-09.csv",
    "data-2022-10.csv",
    "data-2022-11.csv",
    "data-2022-12.csv",

    "data-2023-01.csv",
    "data-2023-02.csv",
    "data-2023-03.csv",
    "data-2023-04.csv",
    "data-2023-05.csv",
    "data-2023-06.csv",
    "data-2023-07.csv",
    "data-2023-08.csv",
    "data-2023-09.csv",
    "data-2023-10.csv",
    "data-2023-11.csv",
    "data-2023-12.csv",

    "data-2024-01.csv",
    "data-2024-02.csv",
    "data-2024-03.csv",
    "data-2024-04.csv",
    "data-2024-05.csv",
    "data-2024-06.csv",
    "data-2024-07.csv",
    "data-2024-08.csv",
    "data-2024-09.csv",
    "data-2024-10.csv",
    "data-2024-11.csv",
    "data-2024-12.csv",

    "data-2025-01.csv",
    "data-2025-02.csv",
    "data-2025-03.csv",
    "data-2025-04.csv",
    "data-2025-05.csv",
    "data-2025-06.csv",
    "data-2025-07.csv",
    "data-2025-08.csv",
    "data-2025-09.csv",
    "data-2025-10.csv",
    "data-2025-11.csv",
    "data-2025-12.csv",

    "data-2026-01.csv",
    "data-2026-02.csv",
    "data-2026-03.csv",
    "data-2026-04.csv",
]



In [3]:
site_columns = [
    "site_id",
    "site_number",
    "longitude",
    "latitude",
    "site_name",
    "domain",
    "road_number",
    "district",
    "municipality",
    "counting_interval_minutes",
    "installation_date",
]

direction_columns = [
    "site_id",
    "direction",
    "direction_description",
]

count_columns = [
    "site_id",
    "direction",
    "traffic_type",
    "start_time",
    "end_time",
    "count",
]

### Reading metadata and external data for modelling normal counts

In [4]:
sites = pd.read_csv(
    raw_folder / "sites.csv",
    header=None,
    names=site_columns,
)

directions = pd.read_csv(
    raw_folder / "richtingen.csv",
    header=None,
    names=direction_columns,
)

public_holidays = pd.read_csv(
    external_folder / "public_holidays.csv"
)

school_holidays_flanders = pd.read_csv(
    external_folder / "school_holidays_flanders.csv"
)

fuel_prices = pd.read_csv(
    external_folder / "fuel_prices_belgium.csv"
)

Converting dates into date format

In [5]:
sites["installation_date"] = pd.to_datetime(
    sites["installation_date"],
    errors="coerce"
)

public_holidays["date"] = pd.to_datetime(
    public_holidays["date"],
    errors="coerce"
)

school_holidays_flanders["start_date"] = pd.to_datetime(
    school_holidays_flanders["start_date"],
    errors="coerce"
)

school_holidays_flanders["end_date"] = pd.to_datetime(
    school_holidays_flanders["end_date"],
    errors="coerce"
)

fuel_prices["date"] = pd.to_datetime(
    fuel_prices["date"],
    errors="coerce"
)

In [6]:
print("Sites:", sites.shape)
print("Directions:", directions.shape)
print("Public holidays:", public_holidays.shape)
print("School holidays:", school_holidays_flanders.shape)
print("Fuel prices:", fuel_prices.shape)

Sites: (151, 11)
Directions: (305, 3)
Public holidays: (56, 2)
School holidays: (20, 3)
Fuel prices: (209, 2)


### Missing values in the monthly count data among the observations recorded for cyclists

During the data loading procedures, missing values were detected for counts in the monthly count datasets. We explore whether there are particular patterns by month and site associated with missing values.

In [7]:
missing_counts_by_month = []

for file_name in count_files:
    df = pd.read_csv(
        raw_folder / file_name,
        header=None,
        names=count_columns,
    )

    
    df = df[df["traffic_type"] == "FIETSERS"].copy()
    df["count"] = pd.to_numeric(df["count"], errors="coerce")

    missing_counts_by_month.append({
        "file_name": file_name,
        "rows": len(df),
        "missing_count": df["count"].isna().sum(),
        "missing_count_share": df["count"].isna().mean(),
        "zero_count": (df["count"] == 0).sum(),
        "zero_count_share": (df["count"] == 0).mean(),
    })

missing_counts_by_month = pd.DataFrame(missing_counts_by_month)

missing_counts_by_month

,file_name,rows,missing_count,missing_count_share,zero_count,zero_count_share
0,data-2022-05.csv,368518,90142,0.244607,147501,0.400255
1,data-2022-06.csv,486830,124142,0.255001,189375,0.388996
2,data-2022-07.csv,632122,117048,0.185167,267164,0.422646
3,data-2022-08.csv,668966,34446,0.051491,324661,0.485318
4,data-2022-09.csv,714104,65592,0.091852,359786,0.503829
5,data-2022-10.csv,767808,29656,0.038624,403696,0.525777
6,data-2022-11.csv,740358,6,0.000008,436504,0.589585
7,data-2022-12.csv,763118,1494,0.001958,492127,0.644890
8,data-2023-01.csv,780396,17086,0.021894,484211,0.620468
9,data-2023-02.csv,720384,15790,0.021919,413726,0.574313


Missing values for counts are more common among the earliest months used in the analysis.

In [8]:
missing_counts_by_month_by_site = []

for file_name in count_files:
    df = pd.read_csv(
        raw_folder / file_name,
        header=None,
        names=count_columns,
    )

    df = df[df["traffic_type"] == "FIETSERS"].copy()
    df["count"] = pd.to_numeric(df["count"], errors="coerce")

    grouped = df.groupby("site_id")["count"]
    group = grouped.agg(
        total_rows="size",
        missing_count=lambda x: x.isna().sum(),
        zero_count=lambda x: (x == 0).sum(),
    ).reset_index()

    group["missing_share"] = group["missing_count"] / group["total_rows"]
    group["zero_share"] = group["zero_count"] / group["total_rows"]
    group["file_name"] = file_name

    missing_counts_by_month_by_site.append(group)

missing_counts_by_month_by_site = pd.concat(missing_counts_by_month_by_site, ignore_index=True)
missing_counts_by_month_by_site

,site_id,total_rows,missing_count,zero_count,missing_share,zero_share,file_name
0,1,5952,0,2102,0.0,0.353159,data-2022-05.csv
1,2,5952,0,3076,0.0,0.516801,data-2022-05.csv
2,3,5952,0,3034,0.0,0.509745,data-2022-05.csv
3,4,5952,0,4167,0.0,0.700101,data-2022-05.csv
4,5,5952,0,3974,0.0,0.667675,data-2022-05.csv
...,...,...,...,...,...,...,...
6491,148,5760,0,2487,0.0,0.431771,data-2026-04.csv
6492,149,5760,0,2192,0.0,0.380556,data-2026-04.csv
6493,150,5760,0,3597,0.0,0.624479,data-2026-04.csv
6494,151,5760,0,3063,0.0,0.531771,data-2026-04.csv


In [ ]:
missing_counts_by_month_by_site = missing_counts_by_month_by_site.sort_values(
    "missing_share",
    ascending=False
)

missing_counts_by_month_by_site.head(20)

,site_id,total_rows,missing_count,zero_count,missing_share,zero_share,file_name
518,127,1050,1050,0,1.000000,0.000000,data-2022-09.csv
144,77,4594,4594,0,1.000000,0.000000,data-2022-06.csv
511,120,4966,4966,0,1.000000,0.000000,data-2022-09.csv
510,119,5162,5162,0,1.000000,0.000000,data-2022-09.csv
509,118,5300,5300,0,1.000000,0.000000,data-2022-09.csv
...,...,...,...,...,...,...,...
3500,118,5952,232,1889,0.038978,0.317372,data-2024-07.csv
3638,118,5952,232,2020,0.038978,0.339382,data-2024-08.csv
4772,3,5952,232,2709,0.038978,0.455141,data-2025-05.csv
3266,20,5760,204,1392,0.035417,0.241667,data-2024-06.csv


In [10]:
missing_counts_by_month_by_site.to_csv(
    diagnostics_folder / "missing_counts_by_site.csv",
    index=False
)

***Possible way to deal with missing values***

Given the distribution of the missing values across months and sites, we try to handle missing values during aggregation from 15-minute intervals to 2-hour intervals. For each 2-hour interval, we store the number of observed and missing 15-minute observations. Intervals with at least 5 out of the expected 8 observations are retained for modelling. For these mostly complete intervals, an adjusted count is computed by scaling the observed sum to an 8-interval equivalent. Intervals with fewer than 5 observed observations are excluded from the modelling dataset.

This approach avoids imputing fully missing site-periods while also avoiding unnecessary loss of mostly complete 2-hour intervals.

### Intervals check

In [11]:
actual_interval_values = []

for file_name in count_files:
    df = pd.read_csv(
        raw_folder / file_name,
        header=None,
        names=count_columns,
    )

    df = df[df["traffic_type"] == "FIETSERS"].copy()

    df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
    df["end_time"] = pd.to_datetime(df["end_time"], errors="coerce")

    df["actual_interval_minutes"] = (
        (df["end_time"] - df["start_time"])
        .dt.total_seconds()
        / 60
    )

    actual_interval_values.append(df[["actual_interval_minutes"]])

actual_interval_values = pd.concat(actual_interval_values, ignore_index=True)

actual_interval_values["actual_interval_minutes"].value_counts(dropna=False).sort_index()

actual_interval_minutes
15.0    37642308
75.0         274
Name: count, dtype: int64

Not all observation intervals are of 15 minutes length.

In [12]:
incorrect_intervals = []

for file_name in count_files:
    df = pd.read_csv(
        raw_folder / file_name,
        header=None,
        names=count_columns,
    )

    df = df[df["traffic_type"] == "FIETSERS"].copy()
    df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
    df["end_time"] = pd.to_datetime(df["end_time"], errors="coerce")
    df["count"] = pd.to_numeric(df["count"], errors="coerce")

    df["actual_interval_minutes"] = (
        (df["end_time"] - df["start_time"]).dt.total_seconds() / 60
    )

    incorrect = df[(df["actual_interval_minutes"] - 15).abs() > 0.1].copy()
    incorrect["file_name"] = file_name

    incorrect_intervals.append(
        incorrect[
            [
                "file_name",
                "site_id",
                "direction",
                "start_time",
                "end_time",
                "actual_interval_minutes",
                "count",
            ]
        ]
    )

incorrect_intervals = pd.concat(incorrect_intervals, ignore_index=True)
incorrect_intervals.head(50)

,file_name,site_id,direction,start_time,end_time,actual_interval_minutes,count
0,data-2023-03.csv,1,IN,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,0.0
1,data-2023-03.csv,1,OUT,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,0.0
2,data-2023-03.csv,2,IN,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,1.0
3,data-2023-03.csv,2,OUT,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,0.0
4,data-2023-03.csv,3,IN,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,0.0
5,data-2023-03.csv,3,OUT,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,1.0
6,data-2023-03.csv,4,IN,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,0.0
7,data-2023-03.csv,4,OUT,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,0.0
8,data-2023-03.csv,5,IN,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,0.0
9,data-2023-03.csv,5,OUT,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,0.0


In [13]:
incorrect_intervals.to_csv(
    diagnostics_folder / "incorrect_intervals.csv",
    index=False
)

**Important**: There was one recording disruption that happened on 26.03.2023 when all the data between 01:45:00 and 03:00:00 was recorded as a single interval due to daylight saving time changes.

### Checking all spring and autumn DST days

In [14]:
spring_dst_dates = [
    "2023-03-26",
    "2024-03-31",
    "2025-03-30",
    "2026-03-29",
]

spring_dst_check = []

for file_name in count_files:
    df = pd.read_csv(
        raw_folder / file_name,
        header=None,
        names=count_columns,
    )

    df = df[df["traffic_type"] == "FIETSERS"].copy()

    df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
    df["end_time"] = pd.to_datetime(df["end_time"], errors="coerce")
    df["count"] = pd.to_numeric(df["count"], errors="coerce")

    df["date"] = df["start_time"].dt.floor("D")
    df["hour"] = df["start_time"].dt.hour
    df["minute"] = df["start_time"].dt.minute

    df["actual_interval_minutes"] = (
        (df["end_time"] - df["start_time"])
        .dt.total_seconds()
        / 60
    )

    temp = df[df["date"].isin(pd.to_datetime(spring_dst_dates))].copy()
    temp["file_name"] = file_name

    spring_dst_check.append(temp)

spring_dst_check = pd.concat(spring_dst_check, ignore_index=True)

In [15]:
spring_dst_hour_summary = (
    spring_dst_check
    .groupby(["date", "hour"])
    .agg(
        rows=("count", "size"),
        unique_sites=("site_id", "nunique"),
        unique_intervals=("actual_interval_minutes", lambda x: sorted(x.dropna().unique())),
        missing_counts=("count", lambda x: x.isna().sum()),
    )
    .reset_index()
)

spring_dst_hour_summary

,date,hour,rows,unique_sites,unique_intervals,missing_counts
0,2023-03-26,0,1096,137,[15.0],24
1,2023-03-26,1,1096,137,"[15.0, 75.0]",24
2,2023-03-26,3,1096,137,[15.0],24
3,2023-03-26,4,1096,137,[15.0],24
4,2023-03-26,5,1096,137,[15.0],24
...,...,...,...,...,...,...
90,2026-03-29,19,1160,145,[15.0],6
91,2026-03-29,20,1160,145,[15.0],6
92,2026-03-29,21,1160,145,[15.0],6
93,2026-03-29,22,1160,145,[15.0],6


In [16]:
spring_dst_hour_summary.to_csv(
    diagnostics_folder / "spring_dst_hour_summary.csv",
    index=False
)

The spring daylight-saving transition is represented inconsistently across years. In 2023, the transition appears as a 75-minute interval from 01:45 to 03:00, while in later years the skipped 02:00 hour is present but contains missing counts. 

In [17]:
autumn_dst_dates = [
    "2022-10-30",
    "2023-10-29",
    "2024-10-27",
    "2025-10-26",
]

autumn_dst_check = []

for file_name in count_files:
    df = pd.read_csv(
        raw_folder / file_name,
        header=None,
        names=count_columns,
    )

    df = df[df["traffic_type"] == "FIETSERS"].copy()

    df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
    df["end_time"] = pd.to_datetime(df["end_time"], errors="coerce")
    df["count"] = pd.to_numeric(df["count"], errors="coerce")

    df["date"] = df["start_time"].dt.floor("D")

    df["actual_interval_minutes"] = (
        (df["end_time"] - df["start_time"])
        .dt.total_seconds()
        / 60
    )

    temp = df[df["date"].isin(pd.to_datetime(autumn_dst_dates))].copy()
    temp["file_name"] = file_name

    autumn_dst_check.append(temp)

autumn_dst_check = pd.concat(autumn_dst_check, ignore_index=True)

In [18]:
autumn_dst_check["is_duplicate_start_time"] = autumn_dst_check.duplicated(
    subset=["date", "site_id", "direction", "start_time"],
    keep=False
)

In [19]:
autumn_dst_duplicates = autumn_dst_check[
    autumn_dst_check["is_duplicate_start_time"]
].copy()

autumn_dst_duplicates = autumn_dst_duplicates.sort_values(
    ["date", "site_id", "direction", "start_time", "end_time"]
)

autumn_dst_duplicates

,site_id,direction,traffic_type,start_time,end_time,count,date,actual_interval_minutes,file_name,is_duplicate_start_time


In [20]:
autumn_dst_check["hour"] = autumn_dst_check["start_time"].dt.hour

autumn_dst_hour_summary = (
    autumn_dst_check
    .groupby(["date", "hour"])
    .agg(
        rows=("count", "size"),
        unique_sites=("site_id", "nunique"),
        unique_intervals=("actual_interval_minutes", lambda x: sorted(x.dropna().unique())),
        missing_counts=("count", lambda x: x.isna().sum()),
        duplicate_start_time_rows=("is_duplicate_start_time", "sum"),
    )
    .reset_index()
)

autumn_dst_hour_summary.to_csv(
    diagnostics_folder / "autumn_dst_hour_summary.csv",
    index=False
)

autumn_dst_hour_summary

,date,hour,rows,unique_sites,unique_intervals,missing_counts,duplicate_start_time_rows
0,2022-10-30,0,1032,129,[15.0],0,0
1,2022-10-30,1,1032,129,[15.0],0,0
2,2022-10-30,2,1032,129,[15.0],0,0
3,2022-10-30,3,1032,129,[15.0],0,0
4,2022-10-30,4,1032,129,[15.0],0,0
...,...,...,...,...,...,...,...
91,2025-10-26,19,1160,145,[15.0],22,0
92,2025-10-26,20,1160,145,[15.0],22,0
93,2025-10-26,21,1160,145,[15.0],22,0
94,2025-10-26,22,1160,145,[15.0],22,0


The autumn daylight saving time transition does not produce duplicated local timestamps in the AWV cyclist count data. All intervals on autumn DST dates remain 15 minutes long, and no duplicated combinations of date, site, direction and start time were found.

## Cleaning and agregating the count data


Only standard 15-minute count observations are used for aggregation. A very small number of non-standard intervals occurs around the spring daylight saving time transition, including one 75-minute interval in 2023. 
These rows are excluded before aggregation because they are not comparable to regular 15-minute observations.


In [21]:
def aggregate_monthly_counts(file_name):

    df = pd.read_csv(
        raw_folder / file_name,
        header=None,
        names=count_columns,
    )

    df = df[df["traffic_type"] == "FIETSERS"].copy()

    df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
    df["end_time"] = pd.to_datetime(df["end_time"], errors="coerce")
    df["count"] = pd.to_numeric(df["count"], errors="coerce")

    df["actual_interval_minutes"] = (
        (df["end_time"] - df["start_time"])
        .dt.total_seconds()
        / 60
    )
    
    df = df[df["actual_interval_minutes"] == 15].copy()

    df["year"] = df["start_time"].dt.year
    df["date"] = df["start_time"].dt.floor("D")
    df["month"] = df["start_time"].dt.month
    df["hour"] = df["start_time"].dt.hour
    df["hour_bin"] = (df["hour"] // 2) * 2
    df["weekday"] = df["start_time"].dt.day_name()

    df_aggregated = (
        df
        .groupby(
            [
                "site_id",
                "direction",
                "year",
                "date",
                "month",
                "weekday",
                "hour_bin",
            ],
            as_index=False
        )
        .agg(
            count=("count", lambda x: x.sum(min_count=1)),
            observed_intervals=("count", "count"),
            total_intervals=("count", "size"),
            missing_intervals=("count", lambda x: x.isna().sum()),
        )
    )

    df_aggregated["missing_share"] = (
        df_aggregated["missing_intervals"] / df_aggregated["total_intervals"]
    )

    return df_aggregated

In [22]:
aggregated_monthly_data = []

for file_name in count_files:
    monthly_aggregated = aggregate_monthly_counts(file_name)
    aggregated_monthly_data.append(monthly_aggregated)

counts_aggregated = pd.concat(aggregated_monthly_data, ignore_index=True)

### Removing 2-hour intervals with less than 5 component 15-min intervals

In [23]:
minimum_observed_intervals = 5

counts_model = counts_aggregated.copy()

counts_model = counts_model[counts_model["count"].notna()].copy()

counts_model = counts_model[
    counts_model["observed_intervals"] >= minimum_observed_intervals
].copy()


### Rescalling incomplete intervals

In [24]:
counts_model["expected_intervals_for_row"] = 8

counts_model["count_rescaled"] = (
    counts_model["count"]
    * counts_model["expected_intervals_for_row"]
    / counts_model["observed_intervals"]
)

counts_model["count_rescaled"] = counts_model["count_rescaled"].round().astype(int)

counts_model["rescaled"] = (
    counts_model["observed_intervals"] < counts_model["expected_intervals_for_row"]
).astype(int)


### Checks

How many rows were kept?

In [25]:
print("Rows before filtering:", counts_aggregated.shape[0])
print("Rows after filtering:", counts_model.shape[0])
print("Rows removed:", counts_aggregated.shape[0] - counts_model.shape[0])

print(
    "Share removed:",
    round(
        (counts_aggregated.shape[0] - counts_model.shape[0])
        / counts_aggregated.shape[0],
        4
    )
)

Rows before filtering: 4705560
Rows after filtering: 4587736
Rows removed: 117824
Share removed: 0.025


Distribution of observed intervals after filtering

In [26]:
counts_model["observed_intervals"].value_counts().sort_index()

observed_intervals
5         65
6         57
7        434
8    4587180
Name: count, dtype: int64

How many rows were rescaled?

In [27]:
counts_model["rescaled"].value_counts()

rescaled
0    4587180
1        556
Name: count, dtype: int64

Rescalling should not change the complete rows

In [28]:
complete_rows_changed = counts_model[
    (counts_model["observed_intervals"] == 8)
    & (counts_model["count"] != counts_model["count_rescaled"])
]

complete_rows_changed.shape[0]

0

### Adding matadata and external source information

In [29]:
def add_metadata_and_external_data(
    counts_model,
    sites,
    directions,
    public_holidays,
    school_holidays_flanders,
    fuel_prices,
):
    counts_model_final = counts_model.copy()

    # Site metadata
    site_variables = [
        "site_id",
        "longitude",
        "latitude",
        "site_name",
        "municipality",
        "district",
        "installation_date",
    ]

    counts_model_final = counts_model_final.merge(
        sites[site_variables],
        on="site_id",
        how="left",
    )

    # Direction metadata
    direction_variables = [
        "site_id",
        "direction",
        "direction_description",
    ]

    counts_model_final = counts_model_final.merge(
        directions[direction_variables],
        on=["site_id", "direction"],
        how="left",
    )

    # Public holidays
    public_holidays_for_merge = public_holidays.copy()

    public_holidays_for_merge["is_public_holiday"] = 1

    public_holidays_for_merge = public_holidays_for_merge[
        ["date", "is_public_holiday", "holiday_name"]
    ].copy()

    counts_model_final = counts_model_final.merge(
        public_holidays_for_merge,
        on="date",
        how="left",
    )

    counts_model_final["is_public_holiday"] = (
        counts_model_final["is_public_holiday"]
        .fillna(0)
        .astype(int)
    )

    counts_model_final["holiday_name"] = (
        counts_model_final["holiday_name"]
        .fillna("No public holiday")
    )

    #School holidays
    counts_model_final["is_school_holiday"] = 0
    counts_model_final["school_holiday_name"] = "No school holiday"

    for _, row in school_holidays_flanders.iterrows():
        in_school_holiday = (
            (counts_model_final["date"] >= row["start_date"])
            & (counts_model_final["date"] <= row["end_date"])
        )

        counts_model_final.loc[
            in_school_holiday,
            "is_school_holiday"
        ] = 1

        counts_model_final.loc[
            in_school_holiday,
            "school_holiday_name"
        ] = row["school_holiday_name"]

    # Fuel prices
    fuel_prices_for_merge = fuel_prices.copy()

    fuel_prices_for_merge = fuel_prices_for_merge.rename(
        columns={
            "petrol_95_eur_per_1000_l": "fuel_price_petrol_95"
        }
    )

    fuel_prices_for_merge = fuel_prices_for_merge[
        ["date", "fuel_price_petrol_95"]
    ].copy()

    fuel_prices_for_merge = fuel_prices_for_merge.sort_values("date")

    # Creating one row per calendar day in the modelling period
    daily_dates = pd.DataFrame({
        "date": pd.date_range(
            counts_model_final["date"].min(),
            counts_model_final["date"].max(),
            freq="D",
        )
    })

    # Assigning each day the most recent available weekly fuel price
    daily_fuel_prices = pd.merge_asof(
        daily_dates.sort_values("date"),
        fuel_prices_for_merge.sort_values("date"),
        on="date",
        direction="backward",
    )

    # Dates are before the first fuel-price date filled using the first available fuel price
    daily_fuel_prices["fuel_price_petrol_95"] = (
        daily_fuel_prices["fuel_price_petrol_95"]
        .bfill()
    )

    counts_model_final = counts_model_final.merge(
        daily_fuel_prices,
        on="date",
        how="left",
    )

    counts_model_final = counts_model_final.sort_values(
        ["site_id", "direction", "date", "hour_bin"]
    ).reset_index(drop=True)

    return counts_model_final

In [30]:
counts_model_final = add_metadata_and_external_data(
    counts_model=counts_model,
    sites=sites,
    directions=directions,
    public_holidays=public_holidays,
    school_holidays_flanders=school_holidays_flanders,
    fuel_prices=fuel_prices,
)

In [31]:
counts_model_final.head(50)

,site_id,direction,year,date,month,weekday,hour_bin,count,observed_intervals,total_intervals,...,site_name,municipality,district,installation_date,direction_description,is_public_holiday,holiday_name,is_school_holiday,school_holiday_name,fuel_price_petrol_95
0,1,IN,2022,2022-05-01,5,Sunday,0,13.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153
1,1,IN,2022,2022-05-01,5,Sunday,2,2.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153
2,1,IN,2022,2022-05-01,5,Sunday,4,1.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153
3,1,IN,2022,2022-05-01,5,Sunday,6,6.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153
4,1,IN,2022,2022-05-01,5,Sunday,8,26.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153
5,1,IN,2022,2022-05-01,5,Sunday,10,28.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153
6,1,IN,2022,2022-05-01,5,Sunday,12,22.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153
7,1,IN,2022,2022-05-01,5,Sunday,14,32.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153
8,1,IN,2022,2022-05-01,5,Sunday,16,20.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153
9,1,IN,2022,2022-05-01,5,Sunday,18,8.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153


In [32]:
counts_model_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 4587736 entries, 0 to 4587735
Data columns (total 27 columns):
 #   Column                      Dtype         
---  ------                      -----         
 0   site_id                     int64         
 1   direction                   str           
 2   year                        int32         
 3   date                        datetime64[us]
 4   month                       int32         
 5   weekday                     str           
 6   hour_bin                    int32         
 7   count                       float64       
 8   observed_intervals          int64         
 9   total_intervals             int64         
 10  missing_intervals           int64         
 11  missing_share               float64       
 12  expected_intervals_for_row  int64         
 13  count_rescaled              int64         
 14  rescaled                    int64         
 15  longitude                   float64       
 16  latitude                    f

In [33]:
counts_model_final.to_csv(
    processed_folder / "counts_model_final.csv",
    index=False
)


### Checks

Rows and columns

In [34]:
print("counts_model rows:", counts_model.shape[0])
print("counts_model_final rows:", counts_model_final.shape[0])
print("columns:", counts_model_final.shape[1])

counts_model rows: 4587736
counts_model_final rows: 4587736
columns: 27


Duplicates

In [35]:
counts_model_final.duplicated(
    subset=["site_id", "direction", "date", "hour_bin"]
).sum()

np.int64(0)

Missing values

In [36]:
counts_model_final.isna().sum()

site_id                           0
direction                         0
year                              0
date                              0
month                             0
weekday                           0
hour_bin                          0
count                             0
observed_intervals                0
total_intervals                   0
missing_intervals                 0
missing_share                     0
expected_intervals_for_row        0
count_rescaled                    0
rescaled                          0
longitude                         0
latitude                          0
site_name                         0
municipality                   5914
district                      27064
installation_date                 0
direction_description             0
is_public_holiday                 0
holiday_name                      0
is_school_holiday                 0
school_holiday_name               0
fuel_price_petrol_95              0
dtype: int64

Missing values are detected for municipality and district variables.

In [37]:
sites.isna().sum()

site_id                      0
site_number                  0
longitude                    0
latitude                     0
site_name                    0
domain                       0
road_number                  3
district                     3
municipality                 1
counting_interval_minutes    0
installation_date            0
dtype: int64

## PART 2: Preparing Data for Deviations Analysis



### Importing packages 

In [38]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

### Using GPS coordinates to recode the geographical columns

According to a brief exploration, we found that the municipality and district columns in the original set contain missing values. This will cause troubles when we include external data by geographical location to detect deviations. Therefore, we will use the exact GPS coordinates (Latitude & Longitude) to verify where these sensors actually are located.

**Initial analysis**

In [39]:
# Creating a combined string column for the exact coordinate pair
counts_model_final['coord_combo'] = counts_model_final['latitude'].astype(str) + ", " + counts_model_final['longitude'].astype(str)

unique_coords_count = counts_model_final['coord_combo'].nunique()
print(f"Total unique physical locations (coordinates): {unique_coords_count}\n")

# Finding unique combinations of ID, Name, and Coordinates to see how they interact
unique_locs = counts_model_final[['site_id', 'site_name', 'coord_combo']].drop_duplicates()

print("MISMATCH ANALYSIS (COORDS VS IDs)")
# Checking if one coordinate has multiple site_ids 
coord_ids_count = unique_locs.groupby('coord_combo')['site_id'].nunique()
mismatched_coords = coord_ids_count[coord_ids_count > 1]

if not mismatched_coords.empty:
    print("One Coordinate with Multiple IDs (Sensors at the exact same spot):")
    print(unique_locs[unique_locs['coord_combo'].isin(mismatched_coords.index)].sort_values('coord_combo').to_string(index=False))

# Checking if one site_name has multiple coordinates 
name_coords_count = unique_locs.groupby('site_name')['coord_combo'].nunique()
mismatched_names = name_coords_count[name_coords_count > 1]

if not mismatched_names.empty:
    print("One Name with Multiple Coordinates (Name re-used in different places):")
    print(unique_locs[unique_locs['site_name'].isin(mismatched_names.index)].sort_values('site_name').to_string(index=False))

Total unique physical locations (coordinates): 147

MISMATCH ANALYSIS (COORDS VS IDs)
One Coordinate with Multiple IDs (Sensors at the exact same spot):
 site_id                         site_name           coord_combo
     116                    Melle teller 2 50.9986113, 3.8040599
     117                    Melle teller 1 50.9986113, 3.8040599
      52                            Brugge     51.22632, 3.24095
     142 TEST Validatie BRUGGE Y2H22022134     51.22632, 3.24095
One Name with Multiple Coordinates (Name re-used in different places):
 site_id              site_name                          coord_combo
      20                 Brugge                    51.17416, 3.20048
      52                 Brugge                    51.22632, 3.24095
      50   Brugge, Sint-Andries                    51.19291, 3.16252
      51   Brugge, Sint-Andries                    51.19281, 3.16252
      98          Geel teller 2                     51.1481, 4.99643
     112          Geel teller 2      

Some coordinates are associated with more than one site ID.

Some site names are reused across nearby but different coordinates. Therefore, site_name alone is not a reliable unique location identifier. For spatial merging and location checks, site_id and coordinates are safer.

### Sites with the same coordinates

In [40]:
sites_to_validate = [116, 117, 52, 142]

sites[
    sites["site_id"].isin(sites_to_validate)
].sort_values("site_id")

,site_id,site_number,longitude,latitude,site_name,domain,road_number,district,municipality,counting_interval_minutes,installation_date
51,52,300024459,3.24095,51.226320,Brugge,Vlaamse Overheid A. Wegen enVerkeer,N3740002,AWV311,Brugge,15,2022-06-01
115,116,300027878,3.80406,50.998611,Melle teller 2,Vlaamse Overheid A. Wegen enVerkeer,N4650001,AWV411,Melle,15,2022-10-01
116,117,300027879,3.80406,50.998611,Melle teller 1,Vlaamse Overheid A. Wegen enVerkeer,N4650001,AWV411,Melle,15,2022-10-01
140,142,300036881,3.24095,51.226320,TEST Validatie BRUGGE Y2H22022134,Vlaamse Overheid A. Wegen enVerkeer,N3740002,AWV311,Brugge,15,2023-10-24


In [41]:
directions[
    directions["site_id"].isin(sites_to_validate)
].sort_values(["site_id", "direction"])

,site_id,direction,direction_description
97,52,IN,Brugge Fietser Brugge
98,52,OUT,Brugge Fietser Koolkerke
235,116,IN,Melle teller 2 Fietser Ovenveldstraat
236,116,OUT,Melle teller 2 Fietser Pontstraat
237,117,IN,Melle teller 1 Fietser Pontstraat
238,117,OUT,Melle teller 1 Fietser Ovenveldstraat
285,142,IN,TEST Validatie BRUGGE Y2H22022134 Fietser IN
286,142,OUT,TEST Validatie BRUGGE Y2H22022134 Fietser UIT


In [42]:
site_time_coverage = (
    counts_model_final[counts_model_final["site_id"].isin(sites_to_validate)]
    .groupby("site_id")
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        number_of_days=("date", "nunique"),
        number_of_rows=("date", "size"),
        directions=("direction", lambda x: sorted(x.dropna().unique())),
    )
    .reset_index()
)

site_time_coverage

,site_id,first_date,last_date,number_of_days,number_of_rows,directions
0,52,2022-05-31,2025-03-06,970,23276,"[IN, OUT]"
1,116,2022-09-30,2026-04-30,1309,31396,"[IN, OUT]"
2,117,2022-09-30,2026-04-30,1309,31396,"[IN, OUT]"
3,142,2023-09-29,2023-11-08,41,984,"[IN, OUT]"


In [43]:
melle_counts = (
    counts_model_final[counts_model_final["site_id"].isin([116, 117])]
    .pivot_table(
        index=["date", "hour_bin", "direction"],
        columns="site_id",
        values="count_rescaled",
        aggfunc="sum"
    )
    .dropna()
)

melle_counts.head()

site_id                        116  117
date       hour_bin direction          
2022-09-30 12       IN         262  536
                    OUT        459  190
           14       IN          85   75
                    OUT          1    1
           16       IN         125   78

In [44]:
melle_counts.corr()

site_id,116,117
site_id,,
116,1.000000,0.780664
117,0.780664,1.000000


In [45]:
brugge_counts= (
    counts_model_final[counts_model_final["site_id"].isin([52, 142])]
    .pivot_table(
        index=["date", "hour_bin", "direction"],
        columns="site_id",
        values="count_rescaled",
        aggfunc="sum"
    )
    .dropna()
)

brugge_counts.head()

,,site_id,52,142
date,hour_bin,direction,,


For Melle, sites 116 and 117 are co-located and active over exactly the same period. Their direction descriptions show that the IN/OUT labels correspond to opposite physical directions across the two counters. They therefore appear to be parallel counters at the same location rather than duplicate records.

For Brugge, site 142 is labelled as a test/validation counter and is active for only 41 days. It shares coordinates with site 52 but does not overlap with site 52 at the date-hour-direction level in the processed data. This suggests that it is a temporary validation or replacement counter rather than a duplicated version of site 52.

### Recoding locations via Geopy

In [46]:
unique_locations = counts_model_final[['latitude', 'longitude']].drop_duplicates().dropna().copy()

# Setting up API
geolocator = Nominatim(user_agent="belgium_traffic_mapper_final")
reverse_geocode = RateLimiter(geolocator.reverse, min_delay_seconds=1)

def get_admin_levels(row):
    coord_str = f"{row['latitude']}, {row['longitude']}"
    try:
        location = reverse_geocode(coord_str, exactly_one=True, language='en')
        address = location.raw.get('address', {})
        
        muni = address.get('municipality') or address.get('town') or address.get('city') or address.get('village') or "Unknown"
        prov = address.get('state', 'Unknown')
        reg = address.get('region', 'Unknown')
        
        return pd.Series([muni, prov, reg])
    except:
        return pd.Series(["Error", "Error", "Error"])


unique_locations[['geopy_municipality', 'geopy_province', 'geopy_region']] = unique_locations.apply(get_admin_levels, axis=1)

# Cleaning up "Province of" text
unique_locations['geopy_province'] = unique_locations['geopy_province'].str.replace('Province of ', '', regex=False)

# Antwerp -> Antwerp Region to distinguish from the city of Antwerp
unique_locations['geopy_province'] = unique_locations['geopy_province'].replace('Antwerp', 'Antwerp Region')


counts_model_adding_geopy = counts_model_final.merge(unique_locations, on=['latitude', 'longitude'], how='left')

In [47]:
print(counts_model_adding_geopy['geopy_region'].value_counts(dropna=False))
print(counts_model_adding_geopy['geopy_province'].value_counts(dropna=False))
print(counts_model_adding_geopy['geopy_municipality'].value_counts(dropna=False))

geopy_region
Unknown             4581822
Brussels-Capital       5914
Name: count, dtype: int64
geopy_province
Flemish Brabant    1090976
Limburg             940071
West Flanders       933696
Antwerp Region      890010
East Flanders       727069
Unknown               5914
Name: count, dtype: int64
geopy_municipality
Laakdal           330724
Heusden-Zolder    251463
Bruges            196166
Kortrijk          157970
Nieuwpoort        135124
                   ...  
Kraainem           21894
Hoeilaart           6032
Brussels            5914
Kortenberg          4712
Opwijk              4380
Name: count, Length: 67, dtype: int64


### Refining Geographic Regions

Since our external data contain many levels of geographic locations, we create 3 columns in total (municipality, province, region) for easier preprocessing. The site in Brussels-Capital region will be coded as Brussels in province column. 5 Flemish provinces are coded as Flanders region.

In [48]:
# Dictionary mapping the 5 Flemish provinces to the Flanders region
province_to_region = {
    'Antwerp Region': 'Flanders',
    'East Flanders': 'Flanders',
    'Flemish Brabant': 'Flanders',
    'Limburg': 'Flanders',
    'West Flanders': 'Flanders'
}

# Fill in 'Flanders' for any row that has one of the 5 provinces
counts_model_adding_geopy['geopy_region'] = counts_model_adding_geopy['geopy_province'].map(province_to_region).fillna(counts_model_adding_geopy['geopy_region'])

# Set province to 'Brussels' where region is 'Brussels-Capital'
counts_model_adding_geopy.loc[counts_model_adding_geopy['geopy_region'] == 'Brussels-Capital', 'geopy_province'] = 'Brussels'


print(counts_model_adding_geopy['geopy_region'].value_counts(dropna=False))
print(counts_model_adding_geopy['geopy_province'].value_counts(dropna=False))
print(counts_model_adding_geopy['geopy_municipality'].value_counts(dropna=False))

geopy_region
Flanders            4581822
Brussels-Capital       5914
Name: count, dtype: int64
geopy_province
Flemish Brabant    1090976
Limburg             940071
West Flanders       933696
Antwerp Region      890010
East Flanders       727069
Brussels              5914
Name: count, dtype: int64
geopy_municipality
Laakdal           330724
Heusden-Zolder    251463
Bruges            196166
Kortrijk          157970
Nieuwpoort        135124
                   ...  
Kraainem           21894
Hoeilaart           6032
Brussels            5914
Kortenberg          4712
Opwijk              4380
Name: count, Length: 67, dtype: int64


In [49]:
pd.set_option("display.max_rows", None)
counts_model_adding_geopy.head(10)

,site_id,direction,year,date,month,weekday,hour_bin,count,observed_intervals,total_intervals,...,direction_description,is_public_holiday,holiday_name,is_school_holiday,school_holiday_name,fuel_price_petrol_95,coord_combo,geopy_municipality,geopy_province,geopy_region
0,1,IN,2022,2022-05-01,5,Sunday,0,13.0,8,8,...,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders
1,1,IN,2022,2022-05-01,5,Sunday,2,2.0,8,8,...,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders
2,1,IN,2022,2022-05-01,5,Sunday,4,1.0,8,8,...,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders
3,1,IN,2022,2022-05-01,5,Sunday,6,6.0,8,8,...,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders
4,1,IN,2022,2022-05-01,5,Sunday,8,26.0,8,8,...,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders
5,1,IN,2022,2022-05-01,5,Sunday,10,28.0,8,8,...,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders
6,1,IN,2022,2022-05-01,5,Sunday,12,22.0,8,8,...,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders
7,1,IN,2022,2022-05-01,5,Sunday,14,32.0,8,8,...,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders
8,1,IN,2022,2022-05-01,5,Sunday,16,20.0,8,8,...,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders
9,1,IN,2022,2022-05-01,5,Sunday,18,8.0,8,8,...,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.82153,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders


### Date Range

As mentioned in our plan, the deviations will be defined in the time frame from 01-05-2025 to 30-04-2026. We recreate the set here. 

In [50]:
counts_model_adding_geopy['date'] = pd.to_datetime(counts_model_adding_geopy['date'])

start_date = pd.to_datetime('2025-05-01')
end_date = pd.to_datetime('2026-04-30')

# Split the data into model development and the new prediction period
model_development_data = counts_model_adding_geopy[counts_model_adding_geopy['date'] < start_date]
prediction_data = counts_model_adding_geopy[(counts_model_adding_geopy['date'] >= start_date) & (counts_model_adding_geopy['date'] <= end_date)]

print(f"Rows in Model Development Data: {len(model_development_data)}")
print(f"Rows in Prediction Data: {len(prediction_data)}\n")

Rows in Model Development Data: 3361867
Rows in Prediction Data: 1225869



### Site analysis and preprocessing

Since we split modelling part into 2 timing phase, we expect new-created site during the final period (2025-2026). We cannot predict behaviors without past data so the determined new sites will be dropped out. 

In [51]:
model_development_sites = set(model_development_data["site_id"].unique())
prediction_sites = set(prediction_data["site_id"].unique())

# Sites in the prediction period that were not present in the model development period
new_prediction_sites = prediction_sites - model_development_sites

print(f"Total unique sites in model development data: {len(model_development_sites)}")
print(f"Total unique sites active in prediction data: {len(prediction_sites)}")
print(f"Number of new prediction-period sites to drop: {len(new_prediction_sites)}")

if len(new_prediction_sites) > 0:
    print(
        "Site IDs being dropped because they are new in the prediction period:",
        sorted(new_prediction_sites),
        "\n"
    )
else:
    print(
        "No new sites found. All sites in the prediction period were present "
        "in the model development data.\n"
    )

Total unique sites in model development data: 141
Total unique sites active in prediction data: 144
Number of new prediction-period sites to drop: 8
Site IDs being dropped because they are new in the prediction period: [np.int64(145), np.int64(146), np.int64(147), np.int64(148), np.int64(149), np.int64(150), np.int64(151), np.int64(152)] 



All the site with IDs from 145 to 152 are new ones. We exclude them in our analysis.

In [52]:
# Filter the dataset for Brussels
brussels_sites = counts_model_adding_geopy[counts_model_adding_geopy['geopy_municipality'] == 'Brussels']

# Get the unique site_ids
unique_brussels_ids = sorted(brussels_sites['site_id'].unique())

print(f"Total number of unique sites in Brussels: {len(unique_brussels_ids)}")
print(f"List of site_ids: {unique_brussels_ids}")

Total number of unique sites in Brussels: 1
List of site_ids: [np.int64(149)]


Notably, the only site existed in Brussels is also a new site, which means the scope of our project now focuses solely on Flanders. 

In [53]:
# Filter the prediction-period data to only include sites that were present in the model development data
prediction_data_clean = prediction_data[
    prediction_data["site_id"].isin(model_development_sites)
].copy()

# Print the final summary to confirm the drop was successful
original_rows = len(prediction_data)
rows_dropped = original_rows - len(prediction_data_clean)

print(f"Original rows in prediction data: {original_rows}")
print(f"Rows remaining after dropping new sites: {len(prediction_data_clean)}")
print(f"Total rows dropped: {rows_dropped}")
print(f"Percentage of data dropped: {(rows_dropped / original_rows) * 100:.2f}%")
print(
    f"Final number of valid unique sites: "
    f"{prediction_data_clean['site_id'].nunique()}"
)

Original rows in prediction data: 1225869
Rows remaining after dropping new sites: 1177575
Total rows dropped: 48294
Percentage of data dropped: 3.94%
Final number of valid unique sites: 136


The last check results in the exclusion of under 5% of original dataset, which is fairly acceptable. 

## Adding external factors data



### Weather data

Weather data are processed to match the temporal and spatial structure of the cycling dataset. Hourly weather observations are assigned to 2-hour time bins and aggregated by site, date, and time bin.

Temperature is averaged within each 2-hour interval. Precipitation, rain, and snowfall are summed over the same interval. A categorical precipitation variable is then created from the precipitation values (with levels: dry periods, light rain, moderate rain, heavy rain, and snow). The final weather dataset is then merged with the cycling data using site_id, date, and hour_bin.

In [54]:
weather_data = pd.read_csv(
    external_folder / "weather_data.csv"
)

weather_data["time"] = pd.to_datetime(weather_data["time"])

weather_data["date"] = weather_data["time"].dt.date
weather_data["date"] = pd.to_datetime(weather_data["date"])

weather_data["hour"] = weather_data["time"].dt.hour
weather_data["hour_bin"] = (weather_data["hour"] // 2) * 2

In [55]:
weather_aggregated = (
    weather_data
    .groupby(["site_id", "date", "hour_bin"], as_index=False)
    .agg(
        temperature_mean=("temperature_2m", "mean"),
        precipitation_sum=("precipitation", "sum"),
        rain_sum=("rain", "sum"),
        snowfall_sum=("snowfall", "sum"),
        wind_speed_mean=("wind_speed_10m", "mean"),
    )
)

In [56]:
prediction_data_with_factors = prediction_data_clean.merge(
    weather_aggregated,
    on=["site_id", "date", "hour_bin"],
    how="left"
)

prediction_data_with_factors.head()

,site_id,direction,year,date,month,weekday,hour_bin,count,observed_intervals,total_intervals,...,fuel_price_petrol_95,coord_combo,geopy_municipality,geopy_province,geopy_region,temperature_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_mean
0,1,IN,2025,2025-05-01,5,Thursday,0,1.0,8,8,...,1.51433,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,15.00,0.0,0.0,0.0,5.95
1,1,IN,2025,2025-05-01,5,Thursday,2,1.0,8,8,...,1.51433,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,13.05,0.0,0.0,0.0,5.80
2,1,IN,2025,2025-05-01,5,Thursday,4,3.0,8,8,...,1.51433,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,11.20,0.0,0.0,0.0,5.10
3,1,IN,2025,2025-05-01,5,Thursday,6,3.0,8,8,...,1.51433,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,10.50,0.0,0.0,0.0,5.95
4,1,IN,2025,2025-05-01,5,Thursday,8,43.0,8,8,...,1.51433,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,14.55,0.0,0.0,0.0,6.00


In [57]:
weather_missing_summary = prediction_data_with_factors[
    [
        "temperature_mean",
        "precipitation_sum",
        "rain_sum",
        "snowfall_sum",
        "wind_speed_mean",
    ]
].isna().mean().sort_values(ascending=False)

weather_missing_summary

temperature_mean     0.0
precipitation_sum    0.0
rain_sum             0.0
snowfall_sum         0.0
wind_speed_mean      0.0
dtype: float64

### Creating categories for precipitation variables

In [58]:
def classify_precipitation(row):
    precipitation = row["precipitation_sum"]
    snowfall = row["snowfall_sum"]

    if pd.isna(precipitation):
        return pd.NA

    if snowfall > 0:
        return "snow"

    if precipitation == 0:
        return "dry"

    if precipitation <= 1:
        return "light_precipitation"

    if precipitation <= 4:
        return "moderate_precipitation"

    return "heavy_precipitation"

In [59]:
prediction_data_with_factors["precipitation_category"] = (
    prediction_data_with_factors.apply(classify_precipitation, axis=1)
)

In [60]:
prediction_data_with_factors["precipitation_category"].value_counts(dropna=False)

precipitation_category
dry                       937578
light_precipitation       181661
moderate_precipitation     40432
snow                       13332
heavy_precipitation         4572
Name: count, dtype: int64

In [61]:
prediction_data_with_factors.head(10)

,site_id,direction,year,date,month,weekday,hour_bin,count,observed_intervals,total_intervals,...,coord_combo,geopy_municipality,geopy_province,geopy_region,temperature_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_mean,precipitation_category
0,1,IN,2025,2025-05-01,5,Thursday,0,1.0,8,8,...,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,15.00,0.0,0.0,0.0,5.95,dry
1,1,IN,2025,2025-05-01,5,Thursday,2,1.0,8,8,...,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,13.05,0.0,0.0,0.0,5.80,dry
2,1,IN,2025,2025-05-01,5,Thursday,4,3.0,8,8,...,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,11.20,0.0,0.0,0.0,5.10,dry
3,1,IN,2025,2025-05-01,5,Thursday,6,3.0,8,8,...,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,10.50,0.0,0.0,0.0,5.95,dry
4,1,IN,2025,2025-05-01,5,Thursday,8,43.0,8,8,...,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,14.55,0.0,0.0,0.0,6.00,dry
5,1,IN,2025,2025-05-01,5,Thursday,10,28.0,8,8,...,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,21.05,0.0,0.0,0.0,3.35,dry
6,1,IN,2025,2025-05-01,5,Thursday,12,19.0,8,8,...,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,25.55,0.0,0.0,0.0,4.70,dry
7,1,IN,2025,2025-05-01,5,Thursday,14,45.0,8,8,...,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,27.10,0.0,0.0,0.0,7.40,dry
8,1,IN,2025,2025-05-01,5,Thursday,16,26.0,8,8,...,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,27.55,0.0,0.0,0.0,5.70,dry
9,1,IN,2025,2025-05-01,5,Thursday,18,19.0,8,8,...,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,27.20,0.0,0.0,0.0,7.80,dry


## Major events data

Here we will import an externally primary self-collected dataset including most of the significant events (strikes, outdoor & indoor music events, running and cycling events) in Flanders during the examining period. 

In [62]:
event_data_path = project_folder / "data" / "external" / "Special_events.xlsx"

strikes = pd.read_excel(event_data_path, sheet_name='transport_strike')
outdoor_music_events = pd.read_excel(event_data_path, sheet_name='music_event')
indoor_music_events = pd.read_excel(event_data_path, sheet_name='indoor_music_event')
sport_events = pd.read_excel(event_data_path, sheet_name='major_sport_event')

**Strikes**

In [63]:
strikes.head()

,event,date,location
0,De Lijn strike,2025-05-20,Flanders
1,SNCB-NMBS,2025-05-20,national
2,SNCB-NMBS,2025-11-24,national
3,SNCB-NMBS,2025-11-25,national
4,SNCB-NMBS,2025-11-26,national


In [64]:
# Make sure both date columns are actual datetime objects so they match perfectly
prediction_data_with_factors['date'] = pd.to_datetime(prediction_data_with_factors['date'])
strikes['date'] = pd.to_datetime(strikes['date'])

# Initialize the new column with 0 (False / No Strike)
prediction_data_with_factors['is_strike'] = 0

In [65]:
# Loop through each strike event and update the 'is_strike' column accordingly
for index, row in strikes.iterrows():
    strike_date = row['date']
    strike_loc = str(row['location']).strip() 
    
    # Condition 1: Date
    date_mask = prediction_data_with_factors['date'] == strike_date
    
    # Condition 2: Region
    if strike_loc.lower() == 'national':
        # If it's national, update all regions to 1 for that date
        prediction_data_with_factors.loc[date_mask, 'is_strike'] = 1
    elif strike_loc.lower() == 'flanders':
        # If it's Flanders, update rows where geopy_region is 'Flanders'
        region_mask = prediction_data_with_factors['geopy_region'] == 'Flanders'
        prediction_data_with_factors.loc[date_mask & region_mask, 'is_strike'] = 1
    else:
        region_mask = prediction_data_with_factors['geopy_province'] == strike_loc
        prediction_data_with_factors.loc[date_mask & region_mask, 'is_strike'] = 1

In [66]:
# Check how many rows were marked as strikes vs non-strikes
strike_counts = prediction_data_with_factors['is_strike'].value_counts()
print(f"Normal days (0): {strike_counts.get(0, 0)}")
print(f"Strike days (1): {strike_counts.get(1, 0)}")

Normal days (0): 1129175
Strike days (1): 48400


**Outdoor music events**

In [114]:
outdoor_music_events.head()

,event name,event type,date,location,time,start_time,end_time
0,Tomorrowland Belgium 2025,music festival,2025-07-18,Boom,12:00-01:00,12:00:00,23:59:00
1,Tomorrowland Belgium 2025,music festival,2025-07-19,Boom,12:00-01:00,00:00:00,01:00:00
2,Tomorrowland Belgium 2025,music festival,2025-07-19,Boom,12:00-01:00,12:00:00,23:59:00
3,Tomorrowland Belgium 2025,music festival,2025-07-20,Boom,12:00-01:00,00:00:00,01:00:00
4,Tomorrowland Belgium 2025,music festival,2025-07-20,Boom,12:00-01:00,12:00:00,23:59:00


In [115]:
# Ensure the date columns are datetime objects
outdoor_music_events['date'] = pd.to_datetime(outdoor_music_events['date'])

# Initialize the new columns
prediction_data_with_factors['is_outdoor_music'] = 0
prediction_data_with_factors['outdoor_music_event_type'] = "No event"  

In [116]:
# Loop through each outdoor music event and update the corresponding columns
for index, row in outdoor_music_events.iterrows():
    # 1. Extract Event Information
    event_date = row['date']
    loc = str(row['location']).strip()
    
    current_event_type = str(row['event type']).strip() 
    
    # 2. Safely parse the start and end times to get the hour (0-23)
    try:
        start_hour = pd.to_datetime(str(row['start_time'])).hour
        end_hour = pd.to_datetime(str(row['end_time'])).hour
        
        if end_hour < start_hour:
            end_hour = 24
    except:
        continue

    # 3. Checking the 3 Conditions
    date_mask = (prediction_data_with_factors['date'] == event_date)
    loc_mask = (prediction_data_with_factors['geopy_municipality'] == loc) | (prediction_data_with_factors['geopy_province'] == loc)
    time_mask = (prediction_data_with_factors['hour_bin'] <= end_hour) & ((prediction_data_with_factors['hour_bin'] + 2) > start_hour)
    
    # The combined mask where ALL conditions are True
    final_match_mask = date_mask & loc_mask & time_mask
    
    # Update BOTH columns for the matching rows!
    prediction_data_with_factors.loc[final_match_mask, 'is_outdoor_music'] = 1
    prediction_data_with_factors.loc[final_match_mask, 'outdoor_music_event_type'] = current_event_type


In [117]:
# Check results
music_counts = prediction_data_with_factors['is_outdoor_music'].value_counts()
print(f"Normal traffic intervals (0): {music_counts.get(0, 0)}")
print(f"Music event intervals (1): {music_counts.get(1, 0)}")

type_counts = prediction_data_with_factors['outdoor_music_event_type'].value_counts()
print(type_counts)

Normal traffic intervals (0): 1175439
Music event intervals (1): 2136
outdoor_music_event_type
No event                     1175439
music festival                   906
theatre/cultural festival        528
cultural city festival           480
cultural parade                  108
arts/music festival               48
food/cultural event               36
city music festival               30
Name: count, dtype: int64


**Indoor music events**

In [118]:
indoor_music_events.head()

,event name,event type,date,location,time,start_time,end_time
0,Tate McRae - Miss Possessive Tour,indoor concert,2025-05-14,Antwerp,18:00-23:59,18:00:00,23:59:00
1,Dotan + Iskander Moon,indoor concert,2025-05-21,Leuven,18:00-23:59,18:00:00,23:59:00
2,The Music of The Lord of the Rings & The Hobbi...,indoor orchestral concert,2025-05-26,Hasselt,18:00-23:00,18:00:00,23:00:00
3,Lords Of The Sound,indoor orchestral concert,2025-06-05,Hasselt,18:00-23:00,18:00:00,23:00:00
4,Dua Lipa - Radical Optimism Tour,indoor concert,2025-06-11,Antwerp,18:00-23:59,18:00:00,23:59:00


In [119]:
# Ensure the date columns are datetime objects
indoor_music_events['date'] = pd.to_datetime(indoor_music_events['date'])

# Initialize the new columns
prediction_data_with_factors['is_indoor_music'] = 0
prediction_data_with_factors['indoor_music_event_type'] = "No event"  

In [120]:
# Loop through each indoor music event and update the corresponding columns
for index, row in indoor_music_events.iterrows():
    # 1. Extract Event Information
    event_date = row['date']
    loc = str(row['location']).strip()
    
    current_event_type = str(row['event type']).strip() 
    
    # 2. Safely parse the start and end times to get the hour (0-23)
    try:
        start_hour = pd.to_datetime(str(row['start_time'])).hour
        end_hour = pd.to_datetime(str(row['end_time'])).hour
        
        if end_hour < start_hour:
            end_hour = 24
    except:
        continue

    # 3. Checking the 3 Conditions
    date_mask = (prediction_data_with_factors['date'] == event_date)
    loc_mask = (prediction_data_with_factors['geopy_municipality'] == loc) | (prediction_data_with_factors['geopy_province'] == loc)
    time_mask = (prediction_data_with_factors['hour_bin'] <= end_hour) & ((prediction_data_with_factors['hour_bin'] + 2) > start_hour)
    
    # The combined mask where ALL conditions are True
    final_match_mask = date_mask & loc_mask & time_mask
    
    # Update BOTH columns for the matching rows!
    prediction_data_with_factors.loc[final_match_mask, 'is_indoor_music'] = 1
    prediction_data_with_factors.loc[final_match_mask, 'indoor_music_event_type'] = current_event_type

In [121]:
# Check results
indoor_music_counts = prediction_data_with_factors['is_indoor_music'].value_counts()
print(f"Normal traffic intervals (0): {indoor_music_counts.get(0, 0)}")
print(f"Music event intervals (1): {indoor_music_counts.get(1, 0)}")

type_counts = prediction_data_with_factors['indoor_music_event_type'].value_counts()
print(type_counts)

Normal traffic intervals (0): 1177287
Music event intervals (1): 288
indoor_music_event_type
No event                     1177287
indoor concert                   150
indoor music festival             54
indoor electronic concert         48
indoor orchestral concert         24
musical concert/show              12
Name: count, dtype: int64


**Sport events**

In [122]:
sport_events.head()

,event,event type,date,location
0,Antwerp Port Epic / Sels Trophy 2025,road cycling race,2025-06-09,Antwerp
1,100KM Dodentocht 2025,mass walking event,2025-08-08,Bornem
2,Renewi Tour 2025 - Stage 3,road cycling stage race,2025-08-22,Aalter
3,Renewi Tour 2025 - Stage 3,road cycling stage race,2025-08-22,Geraardsbergen
4,Renewi Tour 2025 - Final Stage,road cycling stage race,2025-08-24,Leuven


In [123]:
# Ensure the date columns are datetime objects
sport_events['date'] = pd.to_datetime(sport_events['date'])

# Initialize the new columns
prediction_data_with_factors['is_sport_event'] = 0
prediction_data_with_factors['sport_event_type'] = "No event" 

In [124]:
# Loop through each sport event and update the corresponding columns
for index, row in sport_events.iterrows():
    # 1. Extract Event Information
    event_date = row['date']
    loc = str(row['location']).strip()
    
    current_event_type = str(row['event type']).strip() 

    # 2. Check conditions
    date_mask = (prediction_data_with_factors['date'] == event_date)
    loc_mask = (prediction_data_with_factors['geopy_municipality'] == loc) | (prediction_data_with_factors['geopy_province'] == loc)
    
    # The combined mask where BOTH conditions are True
    final_match_mask = date_mask & loc_mask
    
    # Update BOTH columns for the matching rows!
    prediction_data_with_factors.loc[final_match_mask, 'is_sport_event'] = 1
    prediction_data_with_factors.loc[final_match_mask, 'sport_event_type'] = current_event_type

In [125]:
# Check how many rows were marked as events vs normal periods
sport_counts = prediction_data_with_factors['is_sport_event'].value_counts()
print(f"Normal traffic intervals (0): {sport_counts.get(0, 0)}")
print(f"Sports event intervals (1): {sport_counts.get(1, 0)}")

type_counts = prediction_data_with_factors['sport_event_type'].value_counts()
print(type_counts)

Normal traffic intervals (0): 1175971
Sports event intervals (1): 1604
sport_event_type
No event                       1175971
cycling participation event        648
cyclocross race                    432
road cycling race                  264
cyclocross world cup               120
road cycling stage race             96
running event                       44
Name: count, dtype: int64


In [126]:
prediction_data_with_factors.info()

pd.set_option("display.max_columns", None)
prediction_data_with_factors.head(50)

<class 'pandas.DataFrame'>
RangeIndex: 1177575 entries, 0 to 1177574
Data columns (total 44 columns):
 #   Column                      Non-Null Count    Dtype         
---  ------                      --------------    -----         
 0   site_id                     1177575 non-null  int64         
 1   direction                   1177575 non-null  str           
 2   year                        1177575 non-null  int32         
 3   date                        1177575 non-null  datetime64[us]
 4   month                       1177575 non-null  int32         
 5   weekday                     1177575 non-null  str           
 6   hour_bin                    1177575 non-null  int32         
 7   count                       1177575 non-null  float64       
 8   observed_intervals          1177575 non-null  int64         
 9   total_intervals             1177575 non-null  int64         
 10  missing_intervals           1177575 non-null  int64         
 11  missing_share               1177575

,site_id,direction,year,date,month,weekday,hour_bin,count,observed_intervals,total_intervals,missing_intervals,missing_share,expected_intervals_for_row,count_rescaled,rescaled,longitude,latitude,site_name,municipality,district,installation_date,direction_description,is_public_holiday,holiday_name,is_school_holiday,school_holiday_name,fuel_price_petrol_95,coord_combo,geopy_municipality,geopy_province,geopy_region,temperature_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_mean,precipitation_category,is_strike,is_outdoor_music,outdoor_music_event_type,is_indoor_music,indoor_music_event_type,is_sport_event,sport_event_type
0,1,IN,2025,2025-05-01,5,Thursday,0,1.0,8,8,0,0.0,8,1,0,4.456122,50.916183,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.51433,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,15.00,0.0,0.0,0.0,5.95,dry,0,0,No event,0,No event,0,No event
1,1,IN,2025,2025-05-01,5,Thursday,2,1.0,8,8,0,0.0,8,1,0,4.456122,50.916183,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.51433,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,13.05,0.0,0.0,0.0,5.80,dry,0,0,No event,0,No event,0,No event
2,1,IN,2025,2025-05-01,5,Thursday,4,3.0,8,8,0,0.0,8,3,0,4.456122,50.916183,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.51433,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,11.20,0.0,0.0,0.0,5.10,dry,0,0,No event,0,No event,0,No event
3,1,IN,2025,2025-05-01,5,Thursday,6,3.0,8,8,0,0.0,8,3,0,4.456122,50.916183,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.51433,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,10.50,0.0,0.0,0.0,5.95,dry,0,0,No event,0,No event,0,No event
4,1,IN,2025,2025-05-01,5,Thursday,8,43.0,8,8,0,0.0,8,43,0,4.456122,50.916183,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.51433,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,14.55,0.0,0.0,0.0,6.00,dry,0,0,No event,0,No event,0,No event
5,1,IN,2025,2025-05-01,5,Thursday,10,28.0,8,8,0,0.0,8,28,0,4.456122,50.916183,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.51433,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,21.05,0.0,0.0,0.0,3.35,dry,0,0,No event,0,No event,0,No event
6,1,IN,2025,2025-05-01,5,Thursday,12,19.0,8,8,0,0.0,8,19,0,4.456122,50.916183,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.51433,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,25.55,0.0,0.0,0.0,4.70,dry,0,0,No event,0,No event,0,No event
7,1,IN,2025,2025-05-01,5,Thursday,14,45.0,8,8,0,0.0,8,45,0,4.456122,50.916183,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.51433,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,27.10,0.0,0.0,0.0,7.40,dry,0,0,No event,0,No event,0,No event
8,1,IN,2025,2025-05-01,5,Thursday,16,26.0,8,8,0,0.0,8,26,0,4.456122,50.916183,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.51433,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,27.55,0.0,0.0,0.0,5.70,dry,0,0,No event,0,No event,0,No event
9,1,IN,2025,2025-05-01,5,Thursday,18,19.0,8,8,0,0.0,8,19,0,4.456122,50.916183,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,1,Labour Day,0,No school holiday,1.51433,"50.91618331151478, 4.456121776137429",Machelen,Flemish Brabant,Flanders,27.20,0.0,0.0,0.0,7.80,dry,0,0,No event,0,No event,0,No event


In [127]:
event_factors = [
    "is_strike",
    "is_outdoor_music",
    "is_indoor_music",
    "is_sport_event",
]

for col in event_factors:
    print("\n", col)
    print(prediction_data_with_factors[col].value_counts(dropna=False))
    print("Unique values:", prediction_data_with_factors[col].unique()[:10])
    print("Dtype:", prediction_data_with_factors[col].dtype)


 is_strike
is_strike
0    1129175
1      48400
Name: count, dtype: int64
Unique values: [0 1]
Dtype: int64

 is_outdoor_music
is_outdoor_music
0    1175439
1       2136
Name: count, dtype: int64
Unique values: [0 1]
Dtype: int64

 is_indoor_music
is_indoor_music
0    1177287
1        288
Name: count, dtype: int64
Unique values: [0 1]
Dtype: int64

 is_sport_event
is_sport_event
0    1175971
1       1604
Name: count, dtype: int64
Unique values: [0 1]
Dtype: int64


### Save the preprocessed set for deviations modelling

In [128]:
prediction_data_with_factors.to_csv(processed_folder / "prediction_data_with_factors.csv", index=False)